# 01. Bathymetric Data Ingestion, Projected CRS, and Feature EDA
This notebook demonstrates loading real GEBCO_2026 data, metric UTM Zone 18N reprojection, distance validation against WGS84 geodesics, and computing the descriptive statistics, correlations, and spatial autocorrelation (Moran's I).

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_config, fetch_and_reproject_gebco, validate_distance
from src.features import derive_features
from src.analysis import compute_descriptive_stats, compute_correlations, compute_morans_i, compute_pca

cfg = load_config('../config.yaml')
data = fetch_and_reproject_gebco(cfg)
print('Data loaded successfully. Grid shape:', data['z'].shape)


In [ ]:
# Validate projected distance vs WGS84 geodesic
val = validate_distance(cfg['endpoints']['point_A'], cfg['endpoints']['point_B'], crs_projected=cfg['metadata']['crs_projected'])
print(f"Grid Distance: {val['distance_grid_m']:.2f} m")
print(f"Geodesic Distance: {val['distance_geodesic_m']:.2f} m")
print(f"Difference: {val['difference_m']:.2f} m ({val['difference_percent']:.4f}%) - Passed: {val['passed']}")


In [ ]:
# Derive baseline features (slope, roughness, curvature, risk field R)
feat = derive_features(data, cfg)
df_stats = compute_descriptive_stats(feat)
df_stats


In [ ]:
# Correlation Matrices (Pearson and Spearman)
p_corr, s_corr = compute_correlations(feat)
print('--- Pearson Correlation ---')
print(p_corr)
print('\n--- Spearman Correlation ---')
print(s_corr)


In [ ]:
# Spatial Autocorrelation (Moran's I)
morans = {k: compute_morans_i(feat[k]) for k in ['d', 's', 'r', 'c', 'R']}
for k, v in morans.items():
    print(f"Moran's I for {k}: {v:.4f}")


In [ ]:
# Plot depth map and risk field
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
im0 = axes[0].imshow(feat['d'], cmap='Blues')
plt.colorbar(im0, ax=axes[0], label='Depth (m)')
axes[0].set_title('Bathymetric Depth d(x,y)')

im1 = axes[1].imshow(feat['R'], cmap='inferno')
plt.colorbar(im1, ax=axes[1], label='Terrain Risk R')
axes[1].set_title('Composite Terrain Risk R(x,y)')
plt.show()
